# 第2章 RAG 最小構成

外部文書から関連情報を検索し、その内容をプロンプトに混ぜて回答させる **RAG (Retrieval Augmented Generation)** の最小構成を作ります。

**所要時間**: 約30分

## 流れ

1. **Load**: 文書を読み込む
2. **Split**: チャンクに分割
3. **Embed & Store**: ベクトル化して Chroma に保存
4. **Retrieve**: 質問に近いチャンクを検索
5. **Generate**: 検索結果をプロンプトに入れて LLM が回答

## 0. 準備

In [ ]:
import os
from dotenv import load_dotenv

# .env から環境変数を読み込み(第1章と同じ)
load_dotenv()

# 第2章では LLM だけでなく Embedding モデルも使うので、両方のIDを取得する
AWS_REGION = os.environ["AWS_REGION"]
CHAT_MODEL_ID = os.environ["BEDROCK_CHAT_MODEL_ID"]
EMBED_MODEL_ID = os.environ["BEDROCK_EMBED_MODEL_ID"]  # 文章→ベクトル変換用のモデル

print("chat :", CHAT_MODEL_ID)
print("embed:", EMBED_MODEL_ID)

## 1. 文書のロードと分割

`TextLoader` でテキストファイルを読み、`RecursiveCharacterTextSplitter` でチャンクに分けます。

**なぜ分割するのか?**
- LLM の入力トークン数には上限がある(数千〜数十万)。長文をそのまま渡せない
- 文書を細かい単位(チャンク)に分けて検索することで、**質問に関連する部分だけ** をプロンプトに混ぜられる
- 検索精度の面でも、適度な大きさのチャンクの方がベクトル類似度が安定する

**`RecursiveCharacterTextSplitter` の挙動**
- 改行 → 句読点 → スペース → 文字 の順に区切りを探し、できるだけ意味の切れ目で分割しようとする
- `chunk_size`: 1チャンクあたりの最大文字数
- `chunk_overlap`: 隣り合うチャンク間で重複させる文字数。境界をまたいだ情報の取りこぼし防止

**`Document` という型**
- LangChain では、文書は `Document(page_content="本文", metadata={...})` という形で扱う
- `Loader` は `list[Document]` を返し、`Splitter` も `list[Document]` を返す
- このリストの単位がチャンクになる

In [ ]:
# テキストファイル用のローダーと、文字数ベースで再帰的に分割するスプリッターをインポート
# (PDF用の PyPDFLoader、Web用の WebBaseLoader など、ローダーは他にも多数ある)
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# テキストファイルを読み込む。.load() の戻り値は list[Document]
loader = TextLoader("data/sample.md", encoding="utf-8")
raw_docs = loader.load()

# Splitter を準備。chunk_size と chunk_overlap がチューニングポイント
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)

# .split_documents() で Document のリストを受け取り、チャンク化された Document のリストを返す
docs = splitter.split_documents(raw_docs)

# 分割結果を確認。元 1 ファイルから複数チャンクができるのが見える
print(f"分割後のチャンク数: {len(docs)}\n")
for i, d in enumerate(docs):
    print(f"--- chunk {i} ({len(d.page_content)} 字) ---")
    print(d.page_content)  # page_content にチャンクの本文が入る
    print()

## 2. 埋め込みとベクトルDB(Chroma)への保存

- 埋め込み: `BedrockEmbeddings`(Titan Text Embeddings V2)
- ベクトルDB: `Chroma`(`persist_directory` を指定するとローカルに永続化)

**「埋め込み(Embedding)」とは**
テキストを **意味を表す数値ベクトル**(例: 1024次元の浮動小数点配列)に変換する処理です。
似た意味の文章は近い位置のベクトルになるため、「ベクトル間の距離」で類似度検索ができます。

**「ベクトルDB(VectorStore)」とは**
ベクトルと元のテキストをペアで保存し、「このベクトルに近い順に上位N件取って」という検索ができるDB。
- `Chroma` はファイルベースで動く軽量なベクトルDB。本格運用なら Pinecone / OpenSearch / Aurora pgvector など
- `persist_directory` を指定するとそのフォルダにデータが書き出され、再起動後もロードできる

**`Chroma.from_documents()` の挙動**
1. 各 Document の `page_content` を Embedding モデルに渡してベクトル化
2. ベクトルと本文を Chroma に保存
3. すぐ検索に使える `Chroma` インスタンスを返す

> ⚠️ **再実行時の注意**: このセルを2回以上実行すると `./.chroma` に同じドキュメントが重複追加されます。
> 試しに何度も流す場合は、事前にターミナルで `rm -rf 02_rag/.chroma` を実行してから流してください。

In [ ]:
# Bedrock 用の埋め込みモデルクラスと、Chroma ベクトルDBクラスをインポート
from langchain_aws import BedrockEmbeddings
from langchain_chroma import Chroma

# 埋め込みモデルクライアントを準備
# (テキストを渡すとベクトル化して返す。直接使うときは .embed_query() / .embed_documents() がある)
embeddings = BedrockEmbeddings(
    model_id=EMBED_MODEL_ID,    # Titan Text Embeddings V2
    region_name=AWS_REGION,
)

# Document のリストをベクトル化して Chroma に保存し、すぐ検索できる状態の VectorStore を返す
# 内部では「各 Document を embeddings.embed_documents() でベクトル化 → Chroma に追加」を実行
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="./.chroma",  # ベクトルDBの保存先ディレクトリ
)

# DBに登録されたエントリ数を確認(分割後のチャンク数と一致するはず)
print("件数:", len(vectorstore.get()["ids"]))

## 3. 類似検索の動作確認

質問に近いチャンクが返ることを確かめます。

**`similarity_search()` の動作**
1. 引数の質問文を埋め込みモデルでベクトル化
2. Chroma に保存されているベクトルとの距離を計算し、近い順にソート
3. 上位 `k` 件の `Document` を返す

これだけだとまだ LLM を呼んでいません。「文書から候補を抽出する」段階です。

In [ ]:
# 質問に近いチャンクを上位 k=2 件取得する(LLM は使わず、ベクトル類似度のみ)
hits = vectorstore.similarity_search("LCELとは何ですか?", k=2)

# 取得結果を確認。「LCEL」について書かれたチャンクが上位に来るはず
for i, d in enumerate(hits):
    print(f"--- hit {i} ---")
    print(d.page_content)
    print()

## 4. RAGチェーンを組み立てる

LCELで以下を組みます:

```
{ context: retriever → 文字列整形,
  question: そのまま渡す }
        ↓
      prompt
        ↓
       llm
        ↓
    StrOutputParser
```

**Retriever とは**
`vectorstore.as_retriever()` で VectorStore を「検索器(Retriever)」に変換できます。
Retriever は `Runnable` を実装しているので、LCEL チェーンに `|` で組み込めます。
入力: 質問文(文字列) → 出力: 関連 Document のリスト。

**チェーン入口の dict について**
LCEL では dict を渡すと、各キーの値が並列に評価され、結果が dict として次に渡ります。
- `"context": retriever | format_docs` — 質問文を retriever で検索 → format_docs で1つの文字列に連結
- `"question": RunnablePassthrough()` — 入力をそのまま素通り。質問文をそのまま使う

つまり「質問文1つ」を入力すると、`{"context": "<関連チャンク>", "question": "<質問文>"}` という dict に変換され、prompt に渡されます。

**プロンプトに参考情報を注入する形**
system プロンプトに `{context}` の場所を作り、そこに検索結果を埋め込みます。
これにより LLM は「資料に書かれていること」を答える形になり、ハルシネーション(嘘の情報)を抑制できます。

In [ ]:
from langchain_aws import ChatBedrockConverse
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough  # 入力をそのまま通すための Runnable

# LLM 準備(第1章と同じ)
llm = ChatBedrockConverse(
    model=CHAT_MODEL_ID,
    region_name=AWS_REGION,
    temperature=0,
)

# VectorStore を Retriever に変換する。LCELチェーンに組み込めるようになる
# search_kwargs={"k": 3} で「上位3件取得」
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# プロンプト定義。{context} に検索結果、{question} に質問文が入る
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "あなたは社内ドキュメントに基づいて回答するアシスタントです。\n"
     "以下の参考情報のみを根拠に、簡潔に日本語で回答してください。\n"
     "参考情報に答えがない場合は『資料には記載がありません』と答えてください。\n\n"
     "### 参考情報\n{context}"),
    ("human", "{question}"),
])

# Retriever は list[Document] を返すので、文字列に整形する関数を用意する
# (チャンクを2改行区切りで連結。シンプルだが十分実用的)
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# LCEL で RAG チェーン全体を組み立てる
# 入力(質問文字列) → dict 化 → prompt → llm → parser
rag_chain = (
    # dict のキーごとに並列処理される
    {
        # 質問文 → retriever で検索 → format_docs で 1 つの文字列に変換
        "context": retriever | format_docs,
        # 質問文をそのまま {question} に流す
        "question": RunnablePassthrough(),
    }
    | prompt           # dict をフォーマット済みメッセージ列に変換
    | llm              # LLM 推論
    | StrOutputParser()  # AIMessage → 文字列
)

## 5. 質問してみる

In [ ]:
# 質問文字列を invoke に渡すと、内部で retriever 検索 → プロンプト整形 → LLM呼び出し が連鎖実行される
print(rag_chain.invoke("LCELとは何で、何が嬉しいですか?"))

In [ ]:
# 別のテーマで質問。sample.md に書かれている "langchain-aws" の話題が引かれてくる
print(rag_chain.invoke("LangChainでBedrockを使うときのパッケージとクラス名は?"))

### 文書外の質問はどうなる?

プロンプトで「資料にない場合は記載がありませんと答える」よう指示しているので、ハルシネーション抑制が効くか確認します。

In [ ]:
# 文書に含まれない話題。プロンプトの指示通り「資料には記載がありません」と返れば成功
# (LLMが「天気は分かりません」と適当に答える場合もあるが、ハルシネーション抑制の意図は同じ)
print(rag_chain.invoke("今日の東京の天気は?"))

## まとめ

- Loader → Splitter → Embeddings → VectorStore でRAG用のインデックスが作れる
- `vectorstore.as_retriever()` で **検索器** に変換でき、LCELチェーンに組み込める
- 「参考情報」をプロンプトに注入する形にすれば、独自文書を根拠とした回答ができる

次は [03 エージェント](../03_agent/agent.ipynb)。LLMにツールを使わせる仕組みを体験します。